# IAA, Lab 2 : AI and Vision-based navigation
**Students: Emily Baquerizo & Kimberly Beyeler**

In [1]:
import torch
import torch.nn as nn
import torch.onnx
import onnx
import onnxruntime as ort
import numpy as np
from torchvision import models 

## Export ONNX

In [2]:
# Réutilisation de la classe DuckieNet de la partie 1
class DuckieNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        for name, param in backbone.named_parameters():
            if "layer4" not in name and "fc" not in name:
                param.requires_grad = False
        in_features = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

    @staticmethod
    def init_and_load(checkpoint_path: str) -> "DuckieNet":
        model = DuckieNet(pretrained=False)
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()
        return model

# Charger le modèle fine-tuné
CHECKPOINT_DIR = r"C:\Users\Victus\Documents\duckietown\checkpoints"
ONNX_PATH      = CHECKPOINT_DIR + r"\best_finetuned_model.onnx"

model_onnx = DuckieNet.init_and_load(CHECKPOINT_DIR + r"\best_finetuned_model.pt")
model_onnx.eval()

# Export ONNX avec opset 13
torch.onnx.export(
    model_onnx,
    torch.randn(1, 3, 224, 224),
    ONNX_PATH,
    opset_version=13,
    input_names=["image"],
    output_names=["velocities"],
    dynamic_axes={"image": {0: "batch_size"}}
)

# Validation
onnx.checker.check_model(onnx.load(ONNX_PATH))
print("Modele ONNX valide !")
print(f"Sauvegarde : {ONNX_PATH}")

C:\Users\Victus\AppData\Local\Temp\ipykernel_31080\534322328.py:38: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0415 16:33:48.121000 31080 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0415 16:33:50.326000 31080 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `DuckieNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DuckieNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\Victus\anaconda3\envs\IAA\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\Victus\anaconda3\envs\IAA\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\Victus\anaconda3\envs\IAA\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\Victus\anaconda3\envs\IAA\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\Victus\anaconda3\envs\IAA\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
  

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Modele ONNX valide !
Sauvegarde : C:\Users\Victus\Documents\duckietown\checkpoints\best_finetuned_model.onnx


## Vérification PyTorch vs ONNX

In [3]:
test_input = torch.randn(1, 3, 224, 224)

# Prédiction PyTorch
with torch.no_grad():
    pytorch_out = model_onnx(test_input).numpy()

# Prédiction ONNX Runtime
sess = ort.InferenceSession(ONNX_PATH)
onnx_out = sess.run(None, {"image": test_input.numpy()})[0]

print(f"PyTorch : vel_left={pytorch_out[0][0]:.6f}, vel_right={pytorch_out[0][1]:.6f}")
print(f"ONNX    : vel_left={onnx_out[0][0]:.6f},  vel_right={onnx_out[0][1]:.6f}")
print(f"Difference max : {np.abs(pytorch_out - onnx_out).max():.8f}")

if np.abs(pytorch_out - onnx_out).max() < 1e-5:
    print("Les deux modeles sont equivalents !")
else:
    print("Attention : difference significative entre PyTorch et ONNX !")

PyTorch : vel_left=0.619784, vel_right=0.199294
ONNX    : vel_left=0.619784,  vel_right=0.199294
Difference max : 0.00000006
Les deux modeles sont equivalents !


## Exploration YOLOv11

In [5]:
# Explorer le modele YOLOv11 fourni
yolo_model = onnx.load(r"C:\Users\Victus\Documents\heig_vd\iaa\labos\IAA\lab2_part2\tcp-inference-server\models\yolov11_wins_fp32.onnx")

# Architecture
print("Architecture YOLOv11")
print(f"Nombre de noeuds : {len(yolo_model.graph.node)}")

# Inputs
print("\nInputs")
for inp in yolo_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name} : {shape}")

# Outputs
print("\nOutputs")
for out in yolo_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name} : {shape}")

# Classes detectees (depuis driver_node.py)
YOLO_CLASS_NAMES = {
    0: 'Duckie',
    1: 'Duckiebot',
    2: 'Traffic light',
    3: 'QR code',
    4: 'Stop sign',
    5: 'Intersection sign',
    6: 'Signal sign',
}
print(f"\nClasses ({len(YOLO_CLASS_NAMES)}) ===")
for id, name in YOLO_CLASS_NAMES.items():
    print(f"  {id}: {name}")

Architecture YOLOv11
Nombre de noeuds : 320

Inputs
  images : [1, 3, 480, 640]

Outputs
  output0 : [1, 11, 6300]

Classes (7) ===
  0: Duckie
  1: Duckiebot
  2: Traffic light
  3: QR code
  4: Stop sign
  5: Intersection sign
  6: Signal sign


## State Machine

In [6]:
from enum import Enum

class State(Enum):
    DRIVING          = "driving"           # conduite normale
    OBSTACLE_AHEAD   = "obstacle_ahead"    # obstacle détecté devant
    EMERGENCY_STOP   = "emergency_stop"    # arrêt d'urgence
    RECOVERY         = "recovery"          # reprise après arrêt

class StateMachine:
    # Classes YOLO qui déclenchent un arrêt d'urgence
    STOP_CLASSES = {4}           # Stop sign
    # Classes qui nécessitent de ralentir
    SLOW_CLASSES = {0, 1, 2, 5, 6}  # Duckie, Duckiebot, Traffic light, signs

    STOP_SPEED       = 0.0       # vitesse nulle
    SLOW_FACTOR      = 0.5       # réduction de vitesse à 50%
    RECOVERY_SPEED   = 0.15      # vitesse de reprise après arrêt
    STOP_DURATION    = 3.0       # secondes d'arrêt avant reprise
    MIN_BOX_AREA     = 5000      # surface min en pixels² pour considérer l'objet proche

    def __init__(self):
        self.state = State.DRIVING
        self._stop_timer = None

    def update(self, steering, detections):
        """
        Met à jour l'état et retourne les commandes de vitesse.

        Args:
            steering (np.ndarray): [vel_left, vel_right] du modèle
            detections (list): liste de [x1, y1, x2, y2, conf, class_id]

        Returns:
            tuple: (vel_left, vel_right)
        """
        import time
        vel_left, vel_right = float(steering[0][0]), float(steering[0][1])

        # Analyser les détections
        stop_detected  = False
        slow_detected  = False
        for det in detections:
            x1, y1, x2, y2, conf, class_id = det
            area = (x2 - x1) * (y2 - y1)
            if area < self.MIN_BOX_AREA:
                continue  # objet trop loin, ignorer
            if int(class_id) in self.STOP_CLASSES:
                stop_detected = True
            elif int(class_id) in self.SLOW_CLASSES:
                slow_detected = True

        # Transitions d'état
        if self.state == State.DRIVING:
            if stop_detected:
                self.state = State.EMERGENCY_STOP
                self._stop_timer = time.time()
                print(f"[State] DRIVING → EMERGENCY_STOP")
            elif slow_detected:
                self.state = State.OBSTACLE_AHEAD
                print(f"[State] DRIVING → OBSTACLE_AHEAD")

        elif self.state == State.OBSTACLE_AHEAD:
            if stop_detected:
                self.state = State.EMERGENCY_STOP
                self._stop_timer = time.time()
                print(f"[State] OBSTACLE_AHEAD → EMERGENCY_STOP")
            elif not slow_detected:
                self.state = State.DRIVING
                print(f"[State] OBSTACLE_AHEAD → DRIVING")

        elif self.state == State.EMERGENCY_STOP:
            elapsed = time.time() - self._stop_timer
            if elapsed >= self.STOP_DURATION:
                self.state = State.RECOVERY
                print(f"[State] EMERGENCY_STOP → RECOVERY")

        elif self.state == State.RECOVERY:
            if not stop_detected and not slow_detected:
                self.state = State.DRIVING
                print(f"[State] RECOVERY → DRIVING")

        # Calcul des commandes selon l'état
        if self.state == State.DRIVING:
            return vel_left, vel_right

        elif self.state == State.OBSTACLE_AHEAD:
            return vel_left * self.SLOW_FACTOR, vel_right * self.SLOW_FACTOR

        elif self.state == State.EMERGENCY_STOP:
            return self.STOP_SPEED, self.STOP_SPEED

        elif self.state == State.RECOVERY:
            return self.RECOVERY_SPEED, self.RECOVERY_SPEED

# Test de la state machine
sm = StateMachine()
print("State machine initialisée :", sm.state)

# Test sans détection
cmd = sm.update(np.array([[0.22, 0.21]]), [])
print(f"Sans obstacle : {cmd}")

# Test avec Duckie détecté (proche)
cmd = sm.update(np.array([[0.22, 0.21]]), [[100, 100, 200, 300, 0.9, 0]])
print(f"Avec Duckie : {cmd}, state={sm.state}")

# Test avec stop sign
cmd = sm.update(np.array([[0.22, 0.21]]), [[100, 100, 300, 400, 0.9, 4]])
print(f"Avec Stop sign : {cmd}, state={sm.state}")

State machine initialisée : State.DRIVING
Sans obstacle : (0.22, 0.21)
[State] DRIVING → OBSTACLE_AHEAD
Avec Duckie : (0.11, 0.105), state=State.OBSTACLE_AHEAD
[State] OBSTACLE_AHEAD → EMERGENCY_STOP
Avec Stop sign : (0.0, 0.0), state=State.EMERGENCY_STOP
